In [ ]:
# Cell 1: 递归抓取所有 sitemap URL
import time
import requests
import xml.etree.ElementTree as ET

SITEMAP_ROOT = "https://www.retsinformation.dk/eli/sitemap.xml"
NS = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}

def fetch_all_leaf_urls(sitemap_url):
    """返回这个 sitemap 里所有 <loc> URL；如果它是个索引，就递归遍历。"""
    resp = requests.get(sitemap_url)
    resp.raise_for_status()
    root = ET.fromstring(resp.content)
    # 如果包含 <sitemap>，说明是索引
    sitemap_tags = root.findall("sm:sitemap/sm:loc", NS)
    if sitemap_tags:
        urls = []
        for tag in sitemap_tags:
            urls += fetch_all_leaf_urls(tag.text)
            time.sleep(0.1)
        return urls
    # 否则就是叶子，直接拿所有 <url><loc>
    return [loc.text for loc in root.findall("sm:url/sm:loc", NS)]

# 测试：只抓前 1000 条看一下
all_urls = fetch_all_leaf_urls(SITEMAP_ROOT)
print("总 URL 数：", len(all_urls))
print("前 10 条：", all_urls[:10])


总 URL 数： 189510
前 10 条： ['https://www.retsinformation.dk/eli/retsinfo/2025/9567', 'https://www.retsinformation.dk/eli/lta/2025/853', 'https://www.retsinformation.dk/eli/lta/2025/914', 'https://www.retsinformation.dk/eli/retsinfo/2025/9564', 'https://www.retsinformation.dk/eli/retsinfo/2025/9563', 'https://www.retsinformation.dk/eli/ft/20241EB01253', 'https://www.retsinformation.dk/eli/retsinfo/2025/9561', 'https://www.retsinformation.dk/eli/retsinfo/2025/9560', 'https://www.retsinformation.dk/eli/retsinfo/2025/9558', 'https://www.retsinformation.dk/eli/lta/2025/895']


In [2]:
# Cell 2: 过滤出 Lovtidende A/B/C（即“已颁布法令”的 canonical URL）
import re

PAT = re.compile(r"^https://www\.retsinformation\.dk/eli/(lta|ltb|ltc)/\d{4}/\d+$")
law_urls = [u for u in all_urls if PAT.match(u)]
print("匹配到的法令数：", len(law_urls))
print("示例：", law_urls[:5])



匹配到的法令数： 66399
示例： ['https://www.retsinformation.dk/eli/lta/2025/853', 'https://www.retsinformation.dk/eli/lta/2025/914', 'https://www.retsinformation.dk/eli/lta/2025/895', 'https://www.retsinformation.dk/eli/lta/2025/913', 'https://www.retsinformation.dk/eli/lta/2025/877']


In [14]:
pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 10.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 3 (修正版): 抓取、解析、分组并保存为 JSON
import os
import json
import requests
from lxml import etree

def fetch_and_save_law_json_grouped(eli_url: str, save_dir: str = "laws_json"):
    xml_url = eli_url.rstrip("/") + "/xml"
    resp = requests.get(xml_url)
    resp.raise_for_status()
    parser = etree.XMLParser(remove_blank_text=True)
    root = etree.fromstring(resp.content, parser)

    # XPath 辅助函数：不传 namespaces
    def xp(expr, ctx=root):
        return ctx.xpath(expr)

    # 1) 检查 Status
    status = xp('string(//*[local-name()="Meta"]/*[local-name()="Status"])').strip()
    if status != "Valid":
        print(f"❌ 跳过非有效法规: {eli_url}")
        return

    # 2) 收集 meta
    law = {
        "id":             xp('string(//*[local-name()="Meta"]/*[local-name()="Year"])').strip()
                          + "_" +
                          xp('string(//*[local-name()="Meta"]/*[local-name()="Number"])').strip(),
        "title":          xp('string(//*[local-name()="Meta"]/*[local-name()="DocumentTitle"])').strip(),
        "year":           xp('string(//*[local-name()="Meta"]/*[local-name()="Year"])').strip(),
        "number":         xp('string(//*[local-name()="Meta"]/*[local-name()="Number"])').strip(),
        "status":         status,
        "ministry":       xp('string(//*[local-name()="Meta"]/*[local-name()="Ministry"])').strip(),
        "date_published": xp('string(//*[local-name()="Meta"]/*[local-name()="DiesEdicti"])').strip(),
        "signatures":     xp('//*[local-name()="Meta"]/*[local-name()="Signature"]/text()'),
        "concerns":       xp('//*[local-name()="Meta"]/*[local-name()="Concerns"]'
                             '/*[local-name()="Ref_Text"]/text()'),
    }

    # 3) 按 §Paragraf / Stk 分组正文
    structured = []
    for p in xp('//*[local-name()="Paragraf"]'):
        para_num = p.xpath('string(*[local-name()="Explicatus"])').strip()
        para_obj = {"paragraph": para_num, "sections": []}

        for stk in p.xpath('.//*[local-name()="Stk"]'):
            stk_num = stk.xpath('string(*[local-name()="Explicatus"])').strip()
            chars = stk.xpath('.//*[local-name()="Char"]/text()')
            text = " ".join(c.strip() for c in chars if c.strip())
            para_obj["sections"].append({
                "section": stk_num,
                "text": text
            })

        structured.append(para_obj)

    law["structured_text"] = structured

    # 4) 保存为 JSON
    os.makedirs(save_dir, exist_ok=True)
    fname = f"{law['id']}.json"
    path = os.path.join(save_dir, fname)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(law, f, ensure_ascii=False, indent=2)

    print(f"✅ 已保存：{path}")


# 示例：只处理前 3 条
for url in law_urls[:3]:
    fetch_and_save_law_json_grouped(url)


抓取法令： https://www.retsinformation.dk/eli/lta/2025/853
法令 ID: 2025_853
标题: Bekendtgørelse om ikrafttræden af visse bestemmelser i lov om forsvarssamarbejde mellem Danmark og Amerikas Forenede Stater m.v. og § 34, stk. 2, i lov om godkendelse og syn af køretøjer
结构化文本: [{'paragraph': '§ 1.', 'sections': [{'section': '', 'text': '§§ 1-14, §§ 16-23 og § 25 i lov nr. 698 af 20. juni 2025 om forsvarssamarbejde mellem Danmark og Amerikas Forenede Stater m.v. sættes i kraft.'}]}, {'paragraph': '§ 2.', 'sections': [{'section': '', 'text': '§ 34, stk. 2, i lov nr. 774 af 20. juni 2025 om godkendelse og syn af køretøjer sættes i kraft.'}]}, {'paragraph': '§ 3.', 'sections': [{'section': '', 'text': 'Bekendtgørelsen træder i kraft den 1. juli 2025.'}]}]
